# Import Libraries

In [1]:
import sys
#TODO change hardcoding
from concurrent.futures import process
import os
sys.path.append("/home/mahirwar/Projects/LBD/LBD-Analysis/src/data")
import glob
from posixpath import dirname
import re
from turtle import pd
from cv2 import THRESH_BINARY_INV, THRESH_OTSU
import numpy as np
from requests import delete
import cv2
import pyvips as Vips
import pyfiglet
import argparse
import pdb
import skimage.io as io
#from src.utils import vips_utils, normalize
import matplotlib.pyplot as plt
from skimage.io import imread, imsave
import time
from timeit import default_timer as timer
from threading import Thread
from concurrent.futures import ThreadPoolExecutor, wait, ALL_COMPLETED
import subprocess
import geojson
import os
from PIL import Image
import pyvips as Vips
from Reinhard import Reinhard
import pandas as pd
from tqdm import tqdm

## Load Data

In [2]:
dlb_wsi_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/DLB_cases"
pdd_wsi_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases"
dlb_wsi_dir1 = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/DLB_cases/DLB_cases"
wsi_home_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/"

In [3]:
json_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/test_annotations"

In [4]:
f_list = os.listdir(json_dir)
#f_list.remove('.DS_Store')

In [5]:
f_list

['PD110_Syn1_TCx.svs.geojson',
 'PD258_Syn1_PCx.svs.geojson',
 '04_028_Syn1_CG_200x.svs.geojson',
 '19_053_TCx_aSyn_x200.svs.geojson',
 'PD295_Syn1_FCx.svs.geojson',
 'PD268_Syn1_CG.svs.geojson',
 '14_133_FCx_aSyn_x200.svs.geojson',
 '15_134_FCx_aSyn_x200.svs.geojson',
 'PD125_Syn1_PCx.svs.geojson',
 '16_044_PCx_aSyn_x200.svs.geojson']

In [6]:
#filename_correction
import os
f_list_updated_names = []
for file in f_list:
    x = file.replace("SYN1", "Syn1")
    os.rename(os.path.join(json_dir,file),os.path.join(json_dir, x))
    f_list_updated_names.append(x)

In [7]:
f_list_updated_names

['PD110_Syn1_TCx.svs.geojson',
 'PD258_Syn1_PCx.svs.geojson',
 '04_028_Syn1_CG_200x.svs.geojson',
 '19_053_TCx_aSyn_x200.svs.geojson',
 'PD295_Syn1_FCx.svs.geojson',
 'PD268_Syn1_CG.svs.geojson',
 '14_133_FCx_aSyn_x200.svs.geojson',
 '15_134_FCx_aSyn_x200.svs.geojson',
 'PD125_Syn1_PCx.svs.geojson',
 '16_044_PCx_aSyn_x200.svs.geojson']

In [8]:
count_dlb,count_pdd = 0,0
for file in f_list:
    if file.startswith("PD"):
        count_pdd = count_pdd+1
    else:
        count_dlb=count_dlb+1
print("PDD cases: ", count_pdd)
print("DLB cases: ", count_dlb)

PDD cases:  5
DLB cases:  5


In [9]:
#imagenames = sorted(glob.glob(os.path.join(wsi_home_dir, './*/*.svs')))
dlb_imagenames = sorted(glob.glob(os.path.join(dlb_wsi_dir, './*.svs')))
dlb_imagenames1 = sorted(glob.glob(os.path.join(dlb_wsi_dir,"DLB_cases", './*.svs')))
pdd_imagenames = sorted(glob.glob(os.path.join(pdd_wsi_dir, '././*.svs')))

## Generate and save gray matter crops - GT

In [10]:
tilesize = 1024
workers = 10
save_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/WGM_crops_gt1" 
stride = 1024
REF_IMG_PATH = dlb_imagenames[0]
color_codes =  {"White":(0,154,205), "grey":(255, 170, 51),"bg":(128,0,128)}

def normalization(REF_IMG_PATH):
    print("Init Normalization")
    ref_image = Vips.Image.new_from_file(REF_IMG_PATH)
    normalizer = Reinhard()
    normalizer.fit(ref_image)
    return normalizer

def getVipsInfo(vips_img):
    # # Get bounds-x and bounds-y offeset
    vfields = [f.split('.') for f in vips_img.get_fields()]
    vfields = [f for f in vfields if f[0] == 'openslide']
    vfields = dict([('.'.join(k[1:]), vips_img.get('.'.join(k))) for k in vfields])
    return vfields

def get_points_in_contour(x,y,downscaled_w,downscaled_h,cnt,stride, tilesize):
    points = []
    stride = stride
    for x_val in range(x, x+downscaled_w-stride, stride): 
        for y_val in range(y, y + downscaled_h-stride, stride): 
            inside_1 = cv2.pointPolygonTest(cnt, (x_val, y_val), False)
            inside_2 = cv2.pointPolygonTest(cnt, (x_val+tilesize, y_val+tilesize), False)
            inside_3 = cv2.pointPolygonTest(cnt, (x_val, y_val+tilesize), False)
            inside_4 = cv2.pointPolygonTest(cnt, (x_val+tilesize, y_val), False)
            if (inside_1>= 0) and (inside_2>=0) and (inside_3>=0) and (inside_4>=0) : 
                points.append((x_val, y_val)) # points time scale factor print(f'Collected {len(self.points)} points') return self.points 
    return points

def crop_process(i, x, y, vips_orig_img, savesubdir, orig_w, orig_h):   
    print("Crop slide thread Started.", i) 
    savecroppath = os.path.join(savesubdir, f'{filename}_x_{x}_y_{y}.png')
    # row is y, col is x
    if y + tilesize < orig_h and x + tilesize < orig_w:
        # TODO change to vips cropping
        print("---------copying image---------")
        crop = vips_orig_img.crop(x, y, 1024, 1024)
        crop.write_to_file(savecroppath)
        print("Thread Stopped ", i)
        return 1
    else:
        print("Thread Stopped ", i)
        return 0


def tiling(cnt,vips_img, vips_array_copy, filename, class_type, orig_w, orig_h):
    cnt = cnt.astype(np.int)
    x,y,downscaled_w,downscaled_h = cv2.boundingRect(cnt)
    vips_array_copy = cv2.drawContours(vips_array_copy, [cnt],-1, (0,0,255),40)
    vips_array_copy = cv2.rectangle(vips_array_copy,(x, y),(x + downscaled_w,y + downscaled_h),(0,255,0),30)
    points = get_points_in_contour(x,y,downscaled_w,downscaled_h,cnt,stride, tilesize)
    for x, y in points:
        vips_array_copy = cv2.rectangle(vips_array_copy,(x, y),(x + tilesize,y + tilesize),color_codes[class_type],30)
    th = Image.fromarray(vips_array_copy)
    th.thumbnail((1000,1000))

    save_path = os.path.join(save_dir, filename)
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    th.save(os.path.join(save_path,filename.split(".")[0]+".png"))
    
    savesubdir = os.path.join(save_path, class_type)
    if not os.path.exists(savesubdir):
        os.makedirs(savesubdir)

    exe = ThreadPoolExecutor(max_workers=workers)
    futures = [exe.submit(crop_process, i, x_1, y_1, vips_img, savesubdir, orig_w, orig_h) for i, (x_1, y_1) in enumerate(points)]

    done, not_done = wait(futures, return_when=ALL_COMPLETED)
    exe.shutdown()
    print("done tiling")
    #return futures

In [11]:
normalizer = normalization(REF_IMG_PATH)

Init Normalization


In [14]:
annotations_df = pd.DataFrame({"filename": f_list_updated_names,"filepath":[os.path.join(json_dir,x) for x in f_list_updated_names]})

In [15]:
annotations_df

,filename,filepath
0,PD110_Syn1_TCx.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...
1,PD258_Syn1_PCx.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...
2,04_028_Syn1_CG_200x.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...
3,19_053_TCx_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...
4,PD295_Syn1_FCx.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...
5,PD268_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...
6,14_133_FCx_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...
7,15_134_FCx_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...
8,PD125_Syn1_PCx.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...
9,16_044_PCx_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...


In [16]:
def count_annotations_flag(filename, label):
    path = save_dir
    annotated_images = os.listdir(path)
    #annotated_images.remove('.DS_Store')
    filename = filename.replace(".geojson","")
    if filename in annotated_images:
        if os.path.exists(os.path.join(path,filename, label)):
            items = os.listdir(os.path.join(path,filename, label))
            #items = [len(os.listdir(os.path.join(path,filename,i))) for i in wgm_dir]
            return len(items)
        else:
            print("folder does not exist")
            return 0
    else:
        return None

In [17]:
annotations_df["WM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"White"))
annotations_df["GM_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"grey"))
annotations_df["bg_count"]=annotations_df["filename"].apply(lambda l:count_annotations_flag(l,"bg"))

In [18]:
annotations_df

,filename,filepath,WM_count,GM_count,bg_count
0,PD110_Syn1_TCx.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,None,None,None
1,PD258_Syn1_PCx.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,None,None,None
2,04_028_Syn1_CG_200x.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,None,None,None
3,19_053_TCx_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,None,None,None
4,PD295_Syn1_FCx.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,None,None,None
5,PD268_Syn1_CG.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,None,None,None
6,14_133_FCx_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,None,None,None
7,15_134_FCx_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,None,None,None
8,PD125_Syn1_PCx.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,None,None,None
9,16_044_PCx_aSyn_x200.svs.geojson,/gladstone/finkbeiner/steve/work/data/npsad_da...,None,None,None


In [19]:
geofiles_to_annotate =  annotations_df[annotations_df["WM_count"].isna()]["filename"].values
print(geofiles_to_annotate)

['PD110_Syn1_TCx.svs.geojson' 'PD258_Syn1_PCx.svs.geojson'
 '04_028_Syn1_CG_200x.svs.geojson' '19_053_TCx_aSyn_x200.svs.geojson'
 'PD295_Syn1_FCx.svs.geojson' 'PD268_Syn1_CG.svs.geojson'
 '14_133_FCx_aSyn_x200.svs.geojson' '15_134_FCx_aSyn_x200.svs.geojson'
 'PD125_Syn1_PCx.svs.geojson' '16_044_PCx_aSyn_x200.svs.geojson']


In [20]:
geofiles_to_annotate =  annotations_df[annotations_df["WM_count"].isna()]["filename"].values
for geofile in geofiles_to_annotate:
    with open(os.path.join(json_dir,geofile)) as f:
        gj = geojson.load(f)
    filename = geofile.replace(".geojson","")
    features = gj['features']
    print(filename)
    if filename.startswith("PD"):
        img = os.path.join(pdd_wsi_dir, filename)
    elif filename.find("Syn1")!=-1:
        img = os.path.join(dlb_wsi_dir1, filename)
    else:
        img = os.path.join(dlb_wsi_dir, filename)
    #  print(geo_name, len(features))
    vips_img = Vips.Image.new_from_file(img, level=0)
    vips_img = normalizer.transform(vips_img)
    vinfo = getVipsInfo(vips_img)
    orig_w, orig_h = int(vinfo['level[0].width']), int(vinfo['level[0].height'])
    vips_array = np.ndarray(buffer=vips_img.write_to_memory(), dtype=np.uint8, shape=(vips_img.height, vips_img.width, vips_img.bands))
    vips_array = vips_array[:,:,:3]
    vips_array_copy =vips_array.copy()
    for j in features:
        coords = j["geometry"]["coordinates"]
        class_type = j["properties"]["classification"]["names"][0]
        cnt = np.array(coords[0])
        print(len(cnt.shape))
        if (len(cnt.shape)==2) and (class_type=="grey"):
            tiling(cnt,vips_img, vips_array_copy, filename, class_type, orig_w, orig_h)
        else:
            print(filename, cnt.shape)

PD110_Syn1_TCx.svs
2


/tmp/ipykernel_194260/4091114723.py:52: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  cnt = cnt.astype(np.int)


Crop slide thread Started. 0
---------copying image---------
Crop slide thread Started. 1
---------copying image---------
Crop slide thread Started. 2
---------copying image---------
Crop slide thread Started. 3
---------copying image---------
Crop slide thread Started. 4
---------copying image---------
Crop slide thread Started. 5
---------copying image---------
Crop slide thread Started. 6
---------copying image---------
Crop slide thread Started. 7
---------copying image---------
Crop slide thread Started. 8
---------copying image---------
Crop slide thread Started. 9
---------copying image---------
Thread Stopped  6
Crop slide thread Started. 10
---------copying image---------
Thread Stopped  1
Crop slide thread Started. 11
---------copying image---------
Thread Stopped  8
Crop slide thread Started. 12
---------copying image---------
Thread Stopped  4
Crop slide thread Started. 13
---------copying image---------
Thread Stopped  9
Crop slide thread Started. 14
---------copying image

In [11]:
annotations_df =  pd.read_csv("/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/WM_Annotations.csv")